In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [ ]:
# loading and cleaning datasets
df = pd.read_csv("wallets_50+_raw.csv")

clean = df[(df["n_closed"] >= 50) & (df["history_truncated"] == False)].copy()
clean = clean.drop(columns = ["first_end_date", "last_end_date"])

# remove wallets with a suspicious (near-zero) average entry price
clean = clean[clean["avg_entry_price"] >= 0.02].copy()

clean.shape

In [ ]:
# skilled vs other datasets
skilled = clean[(clean["roi"] > 0) & (clean["win_rate_edge"] > 0) & (clean["n_closed"] >= 100)].copy()
other = clean[~clean["wallet"].isin(skilled["wallet"])].copy()

print(len(skilled), len(other), len(clean))

In [ ]:
# plot for ROI
plt.hist(clean["roi"], bins=150, range=(-0.5, 0.5), color="green")
plt.title("ROI of Polymarket Investors")
plt.xlabel("ROI")
plt.ylabel("Number of wallets")
plt.axvline(0, color="red", linestyle="--", label="break-even")
plt.legend()
plt.show()

# share of wallets that made money
(clean["roi"] > 0).mean()

In [ ]:
# KMeans(3)
# both columns are between 0 and 1, so no scaling is needed
ind = ["avg_entry_price", "win_rate"]

model = KMeans(3, n_init=10, random_state=0)
clean["cluster_k3"] = model.fit_predict(clean[ind])

clean["cluster_k3"].value_counts()

In [ ]:
# KMeans(4)
model_v2 = KMeans(4, n_init=10, random_state=0)
clean["cluster_k4"] = model_v2.fit_predict(clean[ind])

clean["cluster_k4"].value_counts()

In [ ]:
# KMeans(5)
model_v3 = KMeans(5, n_init=10, random_state=0)
clean["cluster_k5"] = model_v3.fit_predict(clean[ind])

clean["cluster_k5"].value_counts()

In [ ]:
# what does each cluster look like? (change the column to look at k4 or k5)
clean.groupby("cluster_k3")[["avg_entry_price", "win_rate", "roi", "n_closed"]].median()

In [ ]:
# scatter plot: each dot is a wallet, colored by cluster
plt.scatter(clean["avg_entry_price"], clean["win_rate"], c=clean["cluster_k3"], s=3)
plt.title("Wallet clusters (k=3)")
plt.xlabel("Average entry price")
plt.ylabel("Win rate")
plt.show()

In [ ]:
# median ROI of each cluster
clean.groupby("cluster_k3")["roi"].median().plot.bar(color="green")
plt.title("Median ROI by cluster")
plt.xlabel("Cluster")
plt.ylabel("Median ROI")
plt.show()